# Ensembling engines with `MixSearchEngine`

`MixSearchEngine` runs a list of child engines and synthesizes one answer. It has
two modes, chosen by `MixQueryParams.ensemble_responses`:

- **`False`** (default): each child *retrieves* context, and the mix engine writes a
  single answer from all contexts at once. Cheaper, and the final LLM sees the raw
  evidence.
- **`True`**: each child *answers* independently, and the mix engine reconciles those
  answers. More expensive, but each child reasons inside its own context window
  before anything is merged.

`engine_params` is aligned with `engines`, so one ensemble can drive the graph
engine and the chunk engine with different budgets.

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`, and
optionally `OPENAI_BASE_URL`.

In [ ]:
import os
from pathlib import Path

from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    KnowledgeGraph,
    LocalSearchEngine,
    MixSearchEngine,
    NaiveSearchEngine,
    Settings,
    SimpleChunker,
)
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.search_engine.local_search import LocalParams
from ragu.search_engine.mix_search import MixQueryParams
from ragu.search_engine.naive_search import NaiveSearchParams
from ragu.utils.ragu_utils import read_text_from_files

DATA_DIR = Path("data/en")
QUESTION = "Where did the father of the creator of the C programming language work?"

In [ ]:
Settings.language = "english"
Settings.storage_folder = "ragu_working_dir/mix_search_example"



client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

## Build the graph

The expensive cell. Run once, then re-run the query cells freely.

In [ ]:
knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
    builder_settings=BuilderArguments(),
)
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))

## Assemble the ensemble

The graph engine gets a tighter entity budget, the chunk engine a wider one.
`allow_partial_failures=True` drops a child that raises instead of failing the
whole query; set it to `False` to surface child errors.

In [ ]:
mix = MixSearchEngine(
    llm=llm,
    engines=[
        LocalSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder),
        NaiveSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder),
    ],
    engine_params=[
        LocalParams(top_k=8, use_chunks=False),
        NaiveSearchParams(top_k=15),
    ],
    allow_partial_failures=True,
)

## Mode 1 — ensemble over child *contexts*

Every child retrieves, and one LLM call writes the answer from all the evidence.

In [ ]:
response = await mix.query(QUESTION, MixQueryParams(ensemble_responses=False))

print(response.response)
print(f"\nchild contexts merged: {len(response.retrieval.result.results)}")

## Mode 2 — ensemble over child *answers*

Every child answers on its own first, then the mix engine reconciles them. Seeing
the individual answers next to the synthesis is the useful part: it shows which
child actually carried the final answer.

In [ ]:
response = await mix.query(QUESTION, MixQueryParams(ensemble_responses=True))

print(response.response)
print("\n--- individual child answers ---")
for index, child in enumerate(response.retrieval.result.results, start=1):
    print(f"[{index}] {child.response}\n")

In [ ]:
await knowledge_graph.index.close()